# Hybrid Recommendation System with 5-Slot Architecture

This notebook implements a hybrid recommendation system that combines collaborative filtering (SVD) with a structured 5-slot architecture based on business rules. The goal is to provide a balanced and context-aware set of recommendations that includes replenishment, cross-selling, and new product discovery.

### The 5-Slot Architecture

| Slot | Type | Logic |
|---|---|---|
| **Slot 1** | **Smart Replenishment** | Finds the product the customer is most likely to repurchase, prioritizing frequently bought items that are past their average consumption period. |
| **Slot 2** | **Complementary Product A** | Recommends the most product-line-similar item from a complementary category (same brand, TF-IDF name matching). |
| **Slot 3** | **Complementary Product B** | Same logic as Slot 2, but from a different complementary category to ensure variety. |
| **Slot 4** | **New Product Discovery** | Best SVD-scored new product from a category not yet represented in the recommendation list. |
| **Slot 5** | **Brand Exploration** | Best SVD-scored product from a brand the customer has never purchased, within their price tier and a new category. |                                                        |

### Global Filters

- **Members Only**: The entire model runs only for customers where `is_member == 1`.
- **Price Tier Consistency**: All recommendations are filtered to match the customer's inferred price tier, ensuring a consistent and appropriate brand experience.

### Step 1: Initial Data Preparation

1.  **Filter for Members**: It creates a new dataframe `members_df` containing only records where `is_member == 1`. All subsequent operations use this filtered data.
2.  **Create Proxy Ratings**: It calculates the number of times each customer has purchased each product and applies a logarithmic transformation (`log(count + 1)`). This creates a `proxy_rating` that represents purchase frequency in a scaled way, preventing extreme bulk buyers from dominating the model.
3.  **Prepare for Surprise**: The data is loaded into a `surprise` `Dataset` object, which is the required format for training the SVD model.
4.  **Create Lookups**: Three essential lookup tables are created for fast access later:
    *   `product_metadata`: A table mapping every `product_code` to its brand, category, price tier, and consumption days.
    *   `customer_price_tier`: A table mapping every `customer_id` to their inferred price tier, calculated by finding the `mode` (most frequent value) of the `price_tier` in their purchase history.
    *   `customer_segment`: Maps each `customer_id` to their RFM segment (Champions, Loyal, Hibernating, etc.) — this will appear in the final output.
    *   `all_products`: A simple list of all unique product codes available.

In [7]:
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
import json
import os
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from openai import OpenAI

df = pd.read_csv("/Users/bestek/Desktop/UChicago/Winter'26/ADSP 31017 - ML I/Final Project/clean_data/final_df.csv", parse_dates=['purchase_date'])


# Filter for Members Only 
members_df = df[df['is_member'] == 1].copy()
print(f'Original records: {len(df)} | Member records: {len(members_df)}')

# Create Proxy Ratings (Log of Purchase Count)
purchase_counts = (
    members_df.groupby(['customer_id', 'product_code'])
    .size()
    .reset_index(name='purchase_count')
)
purchase_counts['proxy_rating'] = np.log1p(purchase_counts['purchase_count'])

# Prepare Data for Surprise Library
reader = Reader(rating_scale=(purchase_counts['proxy_rating'].min(), purchase_counts['proxy_rating'].max()))
data = Dataset.load_from_df(purchase_counts[['customer_id', 'product_code', 'proxy_rating']], reader)

# Create Essential Lookups
# Product metadata for easy access
product_metadata = members_df[[
    'product_code', 'product_name', 'brand', 'category_sub', 'price_tier', 'avg_consumption_days'
]].drop_duplicates(subset='product_code').set_index('product_code')

# Customer price tier, inferred from their most frequent purchase tier
customer_price_tier = (
    members_df.groupby('customer_id')['price_tier']
    .agg(lambda x: x.mode()[0])
    .to_frame(name='customer_price_tier')
)

# Customer segment lookup: one row per customer
customer_segment = (
    members_df.groupby('customer_id')['segment']
    .agg(lambda x: x.mode()[0])
    .to_frame(name='segment')
)

# Full list of all products for candidate generation
all_products = members_df['product_code'].unique()

print(f'Total unique customers: {members_df["customer_id"].nunique()}')
print(f'Total unique products: {len(all_products)}')

Original records: 97702 | Member records: 76533
Total unique customers: 19152
Total unique products: 9618


### Step 2: TF-IDF Index

Without name similarity, our complementary recommendation logic would just pick any conditioner from the same brand. With TF-IDF, it picks the conditioner whose name shares the most words with the shampoo the customer bought — which in practice means the same product line.

1. We strip the brand name from each product name (since all products from the same brand share that word, it adds no discriminating power).
2. We build a TF-IDF matrix where each row is a product and each column is a word weight.
3. When we need to find the best complementary match, we calculate the cosine similarity between the source product's vector and all candidate product vectors, and pick the highest-scoring one.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# Build TF-IDF Product Name Similarity Index
# This builds a matrix of numerical vectors from cleaned product names.
# We use it to find the best product-line match for complementary recommendations.
# For example: 'Abeille Royale Shampoo' -> 'Abeille Royale Conditioner'
# rather than just any conditioner from the same brand.

# STEP 1: TF-IDF SETUP
def build_tfidf_index(product_metadata):
    """
    Builds a TF-IDF matrix from cleaned product names (brand removed).
    Returns the matrix and the index mapping product_code to matrix row.

    Why we do this:
    Product names like 'Guerlain Abeille Royale Shampoo' and
    'Guerlain Abeille Royale Conditioner' share the words 'Abeille Royale'.
    TF-IDF cosine similarity captures this overlap and ranks the Abeille Royale
    Conditioner as more similar to the Abeille Royale Shampoo than a random
    conditioner from a different product line.
    """
    tqdm.pandas(desc="Cleaning product names")

    def remove_brand(row):
        return str(row["product_name"]).replace(str(row["brand"]), "").strip()

    product_metadata["name_cleaned"] = product_metadata.progress_apply(remove_brand, axis=1)

    # min_df=2 means a word must appear in at least 2 product names to be included.
    # This removes unique size/batch codes that add noise.
    vectorizer = TfidfVectorizer(min_df=2, stop_words=["ml", "gr", "g", "l", "edp", "edt"])
    tfidf_matrix = vectorizer.fit_transform(product_metadata["name_cleaned"])

    # Map each product_code to its row index in the matrix for fast lookup
    product_indices = {code: i for i, code in enumerate(product_metadata.index)}

    print(f"TF-IDF matrix built: {tfidf_matrix.shape[0]} products, {tfidf_matrix.shape[1]} features.")
    return tfidf_matrix, product_indices

# STEP 2: COMPLEMENTARY PRODUCT FINDER (TF-IDF + Same Brand)
def get_name_similarity_recs(product_code, brand, category_list,
                              product_metadata, tfidf_matrix, product_indices):
    """
    Given a source product, finds products from the same brand that:
    - Belong to one of the complementary categories in category_list
    - Have the highest TF-IDF cosine similarity to the source product name

    This ensures we recommend 'Guerlain Abeille Royale Conditioner'
    rather than just any Guerlain conditioner.
    """
    candidates = product_metadata[
        (product_metadata["brand"] == brand) &
        (product_metadata["category_sub"].isin(category_list))
    ]
    if candidates.empty:
        return []

    source_idx = product_indices.get(product_code)
    if source_idx is None:
        return []

    source_vector = tfidf_matrix[source_idx]

    valid = [(c, product_indices[c]) for c in candidates.index if c in product_indices]
    if not valid:
        return []

    codes, idxs = zip(*valid)
    candidate_vectors = tfidf_matrix[list(idxs)]
    similarities = cosine_similarity(source_vector, candidate_vectors).flatten()

    sim_df = pd.DataFrame({"product_code": list(codes), "similarity": similarities})
    return sim_df.sort_values("similarity", ascending=False)["product_code"].tolist()

tfidf_matrix, product_indices = build_tfidf_index(product_metadata)

Cleaning product names: 100%|██████████| 9618/9618 [00:00<00:00, 118249.15it/s]

TF-IDF matrix built: 9618 products, 2899 features.


### Step 2: Train SVD Model

The SVD model learns a **latent taste profile** for every customer and every product. These are compact numerical vectors that capture hidden preference patterns. When we ask the model to predict a score for a customer-product pair it has never seen, it calculates the similarity between the customer's taste vector and the product's attribute vector.

Key parameters:
- `n_factors=50`: The model learns 50 latent dimensions (taste dimensions) for each user and product. More factors = more expressive model but slower training.
- `n_epochs=20`: The model passes through the training data 20 times to refine its estimates.
- `random_state=42`: Ensures reproducibility — the same model is produced every time you run this cell.

In [11]:
# Train SVD Model
# We use the full dataset to train the model. This ensures the model has the
# maximum amount of data to learn user and item latent factors.
trainset = data.build_full_trainset()
svd_model = SVD(n_factors=50, n_epochs=20, random_state=42)
svd_model.fit(trainset)
print('SVD model trained successfully.')

SVD model trained successfully.


### Step 3: Creating Complemantary Product Mapping Matrix

Instead of manually writing every pair, we let the OpenAI API read all your actual category_sub values and generate a comprehensive, domain-expert-level complementary map automatically. This way the map is grounded in your real data, not guesswork.

In [ ]:
client = OpenAI()

# List of unique sub-categories
unique_categories = df['category_sub'].unique().tolist()

# Created a function to call the OpenAI API to get complemantary product mapping
def generate_complementary_map(categories):
    """Calls OpenAI API to generate a complementary product map."""

    prompt = f"""
    You are a Turkish beauty retail expert. Your task is to create a comprehensive JSON map of complementary products using the provided Turkish category names.

    The list of all available product sub-categories is:
    {", ".join(categories)}

    Generate a JSON object where each key is a sub-category and its value is a list of its complementary sub-categories from the provided list.

    Rules:
    1. The output MUST be a single, valid JSON object.
    2. All keys and values in the JSON MUST use the exact Turkish category names from the list provided. DO NOT translate them.
    3. If a category has no logical complement in the list, do not include it as a key in the JSON.
    4. Be thorough and logical.

    Example of the required JSON format:
    {{
        "Şampuan": ["Saç Kremi", "Saç Maskesi", "Saç Bakım Yağları"],
        "Gündüz Kremi": ["Göz Kremi", "Serumlar", "Tonik"],
        "Fondöten": ["Makyaj Bazı", "Kapatıcı", "Pudra"]
    }}

    JSON output only.
    """

    try:
        print("Calling OpenAI API... (This may take a moment)")
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that returns JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.2,
        )
        
        json_response = response.choices[0].message.content
        complementary_map = json.loads(json_response)
        return complementary_map

    except Exception as e:
        print(f"An error occurred: {e}")
        return None
    
if __name__ == "__main__":
    ai_generated_map = generate_complementary_map(unique_categories)
    
    if ai_generated_map:
        print("\n--- AI-Generated Complementary Map ---")
        print(json.dumps(ai_generated_map, indent=4, ensure_ascii=False))
        
        # Save the map to a file
        with open("/Users/bestek/PycharmProjects/ML_Recommendation_System_Project/complementary_map.json", "w", encoding='utf-8') as f:
            json.dump(ai_generated_map, f, indent=4, ensure_ascii=False)
        print("\nMap successfully saved to complementary_map.json")


Calling OpenAI API... (This may take a moment)

--- AI-Generated Complementary Map ---
{
    "Göz & Dudak Bakım": [
        "Göz Kremi",
        "Dudak Bakım",
        "Serumlar",
        "Göz Makyaj Temizleyici",
        "Takma Kirpik",
        "Göz Kalemi",
        "Eyeliner",
        "Maskara",
        "Göz Farı",
        "Dudak Kalemi",
        "Ruj",
        "Dudak Parlatıcısı"
    ],
    "EDP Parfüm": [
        "Deodorant",
        "Roll-On",
        "Kolonya",
        "Oda Kokusu",
        "Saç Parfümü"
    ],
    "EDT Parfüm": [
        "Deodorant",
        "Roll-On",
        "Kolonya",
        "Oda Kokusu",
        "Saç Parfümü"
    ],
    "Deo Stick": [
        "Deodorant",
        "Roll-On",
        "Kolonya"
    ],
    "Saç Spreyi": [
        "Saç Jölesi",
        "Saç Köpüğü",
        "Saç Serumu",
        "Saç Maskesi",
        "Saç Kremi",
        "Şampuan",
        "Saç Bakım Yağları"
    ],
    "Ağda Bantları": [
        "Ağda Malzemeleri",
        "Roll-On Ağdalar",
 

### Step 4: The 5-Slot Hybrid Recommendation Function

 The `generate_5_slot_recommendations` function orchestrates the entire recommendation logic for a single customer.

1.  **Initialization**: It creates 5 empty slots and sets up trackers for products and categories that have already been used, to ensure diversity.
2.  **Slot 1 (Replenishment)**: It first tries to find a frequently purchased product that is past its consumption period. If it can't, it falls back to finding a recently purchased single-buy product that is past its consumption period. This ensures the most relevant and timely replenishment reminders are prioritized.
3.  **SVD Candidate Pool**: It generates a list of all products the customer has *never* purchased and gets their predicted SVD scores. This list is then filtered to only include products matching the customer's inferred price tier.
4.  **Slots 2 & 3 (Complementary)**: It iterates through the customer's purchase history (most recent first) to find a product with a defined complementary relationship. It then searches the SVD candidate list for the best-scoring products from those complementary categories.
5.  **Slot 4 (New Product)**: It finds the best-scoring SVD product from a category that has not been used in any of the previous slots.
6.  **Slot 5 (Brand Exploration)**: It finds the best-scoring SVD product from a brand the customer has never tried before, ensuring it is still within their price tier.
7.  **Fill Empty Slots**: If any of the 5 slots remain empty after all the specific logic has run (e.g., no valid replenishment or complementary products were found), the remaining slots are filled with the next-best new products from the SVD list, ensuring a full top-5 is always returned.

In [12]:
# The 5-Slot Hybrid Recommendation Function

with open("/Users/bestek/PycharmProjects/ML_Recommendation_System_Project/complementary_map.json", 'r', encoding='utf-8') as f:
    COMPLEMENTARY_MAP = json.load(f)
print(f'Successfully loaded {len(COMPLEMENTARY_MAP)} categories from complementary_map.json')

def generate_5_slot_recommendations(
    customer_id, today,
    members_df, purchase_counts, product_metadata,
    customer_price_tier, all_products, svd_model,
    tfidf_matrix, product_indices, COMPLEMENTARY_MAP
):
    """
    Generates a 5-slot recommendation list for a single member customer.

    Slot 1: Smart Replenishment
        - Looks at products bought more than once, sorted by frequency.
        - Recommends the first one where time since last purchase >= avg_consumption_days.
        - Falls back to single-purchase products sorted by recency.
        - If nothing qualifies, this slot is left for the fallback pool.

    Slots 2 & 3: Complementary Products (Same Brand, TF-IDF Name Similarity)
        - Iterates through purchase history (most frequent first) to find a product
          with a mapping in the COMPLEMENTARY_MAP.
        - Uses TF-IDF cosine similarity to find the best product-line match
          within the same brand and a complementary category.
        - Each slot must be from a different category (category exclusivity).

    Slot 4: New Product Discovery
        - Best SVD-scored new product from a category not yet used in Slots 1-3.
        - Must be within the customer's price tier.

    Slot 5: Brand Exploration
        - Best SVD-scored new product from a brand the customer has never purchased.
        - Must be within the customer's price tier.
        - Must be from a category not yet used in Slots 1-4.

    Fallback: Any empty slots are filled with the next-best diverse SVD new products.
    """
    slots = {i: None for i in range(1, 6)}
    used_products = set()
    used_categories = set()

    # --- Retrieve customer data ---
    try:
        customer_history = members_df[members_df["customer_id"] == customer_id]
        cust_price_tier = customer_price_tier.loc[customer_id, "customer_price_tier"]
    except KeyError:
        return []

    customer_brands = set(customer_history["brand"].unique())

    # SLOT 1: SMART REPLENISHMENT
    # Priority A: Products bought more than once, sorted by frequency (highest first)
    cust_purchase_counts = purchase_counts[purchase_counts["customer_id"] == customer_id]
    multi_purchase = cust_purchase_counts[cust_purchase_counts["purchase_count"] > 1].sort_values(
        "purchase_count", ascending=False
    )

    for _, row in multi_purchase.iterrows():
        prod_code = row["product_code"]
        if prod_code not in product_metadata.index:
            continue
        last_purchase = customer_history[customer_history["product_code"] == prod_code]["purchase_date"].max()
        days_since = (today - last_purchase).days
        consumption_days = product_metadata.loc[prod_code, "avg_consumption_days"]
        if days_since >= consumption_days:
            cat = product_metadata.loc[prod_code, "category_sub"]
            slots[1] = {
                "product_code": prod_code,
                "brand": product_metadata.loc[prod_code, "brand"],
                "category_sub": cat,
                "price_tier": product_metadata.loc[prod_code, "price_tier"],
                "score": float(row["purchase_count"]),
                "recommendation_type": "Replenishment (Repeat Purchase)"
            }
            used_products.add(prod_code)
            used_categories.add(cat)
            break

    # Priority B: Single-purchase products, sorted by recency
    if not slots[1]:
        single_purchase_history = customer_history[
            ~customer_history["product_code"].isin(multi_purchase["product_code"])
        ].sort_values("purchase_date", ascending=False)

        for _, row in single_purchase_history.iterrows():
            prod_code = row["product_code"]
            if prod_code not in product_metadata.index:
                continue
            days_since = (today - row["purchase_date"]).days
            consumption_days = product_metadata.loc[prod_code, "avg_consumption_days"]
            if days_since >= consumption_days:
                cat = product_metadata.loc[prod_code, "category_sub"]
                slots[1] = {
                    "product_code": prod_code,
                    "brand": product_metadata.loc[prod_code, "brand"],
                    "category_sub": cat,
                    "price_tier": product_metadata.loc[prod_code, "price_tier"],
                    "score": 1.0,
                    "recommendation_type": "Replenishment (Single Purchase)"
                }
                used_products.add(prod_code)
                used_categories.add(cat)
                break

    # SVD CANDIDATE POOL (New Products Only, Tier-Filtered)
    purchased_prods = customer_history["product_code"].unique()
    svd_candidates = np.setdiff1d(all_products, purchased_prods)

    svd_predictions = [svd_model.predict(customer_id, prod) for prod in svd_candidates]
    svd_predictions.sort(key=lambda x: x.est, reverse=True)

    # Filter by customer's price tier
    tier_filtered_svd = [
        p for p in svd_predictions
        if p.iid in product_metadata.index and
        product_metadata.loc[p.iid, "price_tier"] == cust_price_tier
    ]

    # SLOTS 2 & 3: COMPLEMENTARY PRODUCTS (TF-IDF + SAME BRAND)
    # Iterate through purchase history (most recent first) to find a product
    # with a complementary mapping.
    history_by_recency = customer_history.sort_values("purchase_date", ascending=False)
    slots_23_filled = 0

    for _, hist_row in history_by_recency.iterrows():
        if slots_23_filled >= 2:
            break
        source_cat = hist_row["category_sub"]
        source_brand = hist_row["brand"]
        comp_cats = COMPLEMENTARY_MAP.get(source_cat)
        if not comp_cats:
            continue

        # Find available complementary categories (not yet used)
        available_comp_cats = [c for c in comp_cats if c not in used_categories]
        if not available_comp_cats:
            continue

        # Use TF-IDF to find best product-line match within same brand
        sim_recs = get_name_similarity_recs(
            hist_row["product_code"], source_brand, available_comp_cats,
            product_metadata, tfidf_matrix, product_indices
        )

        for rec_code in sim_recs:
            if rec_code in used_products:
                continue
            if rec_code not in product_metadata.index:
                continue
            rec_cat = product_metadata.loc[rec_code, "category_sub"]
            if rec_cat in used_categories:
                continue
            slot_num = 2 if slots_23_filled == 0 else 3
            slots[slot_num] = {
                "product_code": rec_code,
                "brand": product_metadata.loc[rec_code, "brand"],
                "category_sub": rec_cat,
                "price_tier": product_metadata.loc[rec_code, "price_tier"],
                "score": None,
                "recommendation_type": "Complementary (TF-IDF Name Match)"
            }
            used_products.add(rec_code)
            used_categories.add(rec_cat)
            slots_23_filled += 1
            break

    # SLOT 4: NEW PRODUCT DISCOVERY
    for pred in tier_filtered_svd:
        if pred.iid in used_products:
            continue
        cat = product_metadata.loc[pred.iid, "category_sub"]
        if cat in used_categories:
            continue
        slots[4] = {
            "product_code": pred.iid,
            "brand": product_metadata.loc[pred.iid, "brand"],
            "category_sub": cat,
            "price_tier": product_metadata.loc[pred.iid, "price_tier"],
            "score": round(pred.est, 4),
            "recommendation_type": "New Product Discovery"
        }
        used_products.add(pred.iid)
        used_categories.add(cat)
        break

    # SLOT 5: BRAND EXPLORATION
    for pred in tier_filtered_svd:
        if pred.iid in used_products:
            continue
        prod_brand = product_metadata.loc[pred.iid, "brand"]
        if prod_brand in customer_brands:
            continue  # Skip brands the customer already knows
        cat = product_metadata.loc[pred.iid, "category_sub"]
        if cat in used_categories:
            continue  # Enforce category exclusivity across all 5 slots
        slots[5] = {
            "product_code": pred.iid,
            "brand": prod_brand,
            "category_sub": cat,
            "price_tier": product_metadata.loc[pred.iid, "price_tier"],
            "score": round(pred.est, 4),
            "recommendation_type": "Brand Exploration"
        }
        used_products.add(pred.iid)
        used_categories.add(cat)
        break

    # FALLBACK: Fill any remaining empty slots
    for pred in tier_filtered_svd:
        if all(v is not None for v in slots.values()):
            break
        if pred.iid in used_products:
            continue
        cat = product_metadata.loc[pred.iid, "category_sub"]
        if cat in used_categories:
            continue
        for slot_num in range(1, 6):
            if slots[slot_num] is None:
                slots[slot_num] = {
                    "product_code": pred.iid,
                    "brand": product_metadata.loc[pred.iid, "brand"],
                    "category_sub": cat,
                    "price_tier": product_metadata.loc[pred.iid, "price_tier"],
                    "score": round(pred.est, 4),
                    "recommendation_type": "Fallback (SVD)"
                }
                used_products.add(pred.iid)
                used_categories.add(cat)
                break

    return [{"slot": k, **v} for k, v in sorted(slots.items()) if v is not None]

Successfully loaded 133 categories from complementary_map.json


In [13]:
# Generate Recommendations for a Sample of Customers

# Set the reference date.
# For production, replace with pd.Timestamp.today() to always use the current date.
TODAY = pd.to_datetime('2026-02-18')

all_customer_ids = members_df['customer_id'].unique()

# We sample 10 customers to quickly validate the pipeline.
# When ready to run the full dataset, replace the line below with:
#   sample_ids = all_customer_ids
sample_ids = pd.Series(all_customer_ids).sample(n=min(10, len(all_customer_ids)), random_state=42).values

all_recommendations = []

for cust_id in tqdm(sample_ids, desc='Generating Recommendations', unit='customer'):
    recs = generate_5_slot_recommendations(
        customer_id=cust_id,
        today=TODAY,
        members_df=members_df,
        purchase_counts=purchase_counts,
        product_metadata=product_metadata,
        customer_price_tier=customer_price_tier,
        all_products=all_products,
        svd_model=svd_model,
        tfidf_matrix=tfidf_matrix,
        product_indices=product_indices,
        COMPLEMENTARY_MAP=COMPLEMENTARY_MAP
    )

    # Get customer-level metadata for the output
    segment = customer_segment.loc[cust_id, 'segment'] if cust_id in customer_segment.index else 'Unknown'
    price_tier = customer_price_tier.loc[cust_id, 'customer_price_tier'] if cust_id in customer_price_tier.index else 'Unknown'

    for rec in recs:
        all_recommendations.append({
            'customer_id':         cust_id,
            'segment':             segment,
            'customer_price_tier': price_tier,
            **rec
        })

recommendations_df = pd.DataFrame(all_recommendations)

print(f"\nDone. Total recommendation rows: {len(recommendations_df)}")
print(f"\nRecommendation type breakdown:")
print(recommendations_df['recommendation_type'].value_counts())

Generating Recommendations:   0%|          | 0/10 [00:00<?, ?customer/s]

Generating Recommendations: 100%|██████████| 10/10 [00:01<00:00,  7.88customer/s]


Done. Total recommendation rows: 50

Recommendation type breakdown:
recommendation_type
Complementary (TF-IDF Name Match)    12
New Product Discovery                10
Brand Exploration                    10
Replenishment (Single Purchase)       9
Fallback (SVD)                        9
Name: count, dtype: int64


### Step 5: Explanation of the Output

Each row in `recommendations_df` represents one recommended product for one customer. The columns are:

| Column | Description |
|---|---|
| `customer_id` | The unique customer identifier |
| `segment` | The customer's RFM segment (Champions, Loyal Customers, Hibernating, etc.) |
| `customer_price_tier` | The customer's inferred price tier (Luxury, Premium, Mid-Tier, Economy) |
| `slot` | Which of the 5 recommendation slots this product fills (1-5) |
| `recommendation_type` | Why this product was recommended (Replenishment, Complementary, New Product Discovery, Brand Exploration, or Fallback) |
| `product_code` | The product code of the recommended product |
| `brand` | The brand of the recommended product |
| `category_sub` | The sub-category of the recommended product |
| `price_tier` | The price tier of the recommended product |
| `score` | The SVD predicted score (higher = stronger preference signal). Null for Complementary slots, which are selected by TF-IDF similarity rather than SVD score. |


In [14]:
recommendations_df

,customer_id,segment,customer_price_tier,slot,product_code,brand,category_sub,price_tier,score,recommendation_type
0,T49900,Potential Loyalists,Economy,1,1263-2262,Natural Colors,Natural Boyalar,Economy,1.0000,Replenishment (Single Purchase)
1,T49900,Potential Loyalists,Economy,2,67531,Natural Colors,Peroksitler,Economy,NaN,Complementary (TF-IDF Name Match)
2,T49900,Potential Loyalists,Economy,3,53377,Herbal,Şampuan,Economy,1.0217,Fallback (SVD)
3,T49900,Potential Loyalists,Economy,4,123067,Arko,El Bakım,Economy,1.0791,New Product Discovery
4,T49900,Potential Loyalists,Economy,5,120867,Carefree,Günlük Pedler,Economy,1.0734,Brand Exploration
5,T38547,Hibernating,Luxury,1,2145-2607,YSL,Fondöten,Luxury,1.0000,Replenishment (Single Purchase)
6,T38547,Hibernating,Luxury,2,2217-2954,YSL,Kapatıcı,Luxury,NaN,Complementary (TF-IDF Name Match)
7,T38547,Hibernating,Luxury,3,114,Clinique,Göz Makyaj Temizleyici,Luxury,1.0532,Fallback (SVD)
8,T38547,Hibernating,Luxury,4,1090-319,Lancome,Maskara,Luxury,1.1087,New Product Discovery
9,T38547,Hibernating,Luxury,5,127557,La Prairie,Temizleme Jeli,Luxury,1.0634,Brand Exploration


In [15]:
# ── Optional: Save the Recommendations to CSV ─────────────────────────────
recommendations_df.to_csv("/Users/bestek/Desktop/UChicago/Winter'26/ADSP 31017 - ML I/Final Project/clean_data/recommendations_output.cs", index=False)
print('Recommendations saved to recommendations_output.csv')

Recommendations saved to recommendations_output.csv
